# Setpoint Feedforward (Anticipation de la consigne)

Simulation of the GEL-7063 exercise model `ex_anticipation_consigne.slx`
using the python-control library.

The control system is a 2-DOF structure with model-based setpoint
feedforward:

$$
U(s) = G_{ff}(s) R(s) + G_c(s) \left( G_r(s) R(s) - Y(s) \right)
$$

$$
Y(s) = G_p(s) U(s) + D(s)
$$

where:
 - $G_p(s)$ : plant (process)
 - $G_m(s)$ : model of the plant, used to design $G_{ff}(s)$ and $G_r(s)$
 - $G_c(s)$ : PI feedback controller
 - $G_{ff}(s)$ : setpoint feedforward
 - $G_r(s)$ : reference model (desired setpoint response)
 - $D(s)$ : output disturbance

In [ ]:
import control as con
import matplotlib.pyplot as plt
import numpy as np

## Plant

$$
G_p(s) = \frac{3(-2s + 1)}{(10s + 1)(5s + 1)} e^{-4s}
$$

python-control transfer functions are rational, so the time delay is
kept separate from the rational part $G_{p,0}(s)$. For simulation, the
models are converted to discrete time, where the delay is represented
exactly as a shift of $\theta / T_s$ samples (see
[Discretization](#Discretization)).

In [ ]:
s = con.tf('s')

# Plant parameters
K_p = 3.0      # gain
T_1 = 10.0     # dominant time constant (s)
T_2 = 5.0      # secondary time constant (s)
T_z = 2.0      # RHP zero time constant (s)
theta = 4.0    # time delay (s)

# Rational part of plant
G_p0 = K_p * (-T_z * s + 1) / ((T_1 * s + 1) * (T_2 * s + 1))
print(G_p0)

## PI Feedback Controller

$$
G_c(s) = K_c \frac{T_i s + 1}{T_i s}
$$

In [ ]:
# Controller parameters
K_c = 0.21
T_i = 10.0

G_c = K_c * (T_i * s + 1) / (T_i * s)
print(G_c)

## Setpoint Feedforward and Reference Model

### Reference Filter

The reference filter $F(s)$ specifies the desired setpoint tracking
dynamics (user choice):

$$
F(s) = \frac{1}{(5s + 1)^2}
$$

In [ ]:
# Reference filter parameters
T_F = 5.0      # filter time constant (s)

F = 1 / (T_F * s + 1) ** 2
print(F)

### Plant Model Decomposition

The model of the plant is assumed to be perfect, $G_m(s) = G_p(s)$. It
is decomposed into invertible and non-invertible parts:

$$
G_m(s) = \tilde{G}_m(s) \bar{G}_m(s)
= \frac{3}{(10s + 1)(5s + 1)} \cdot (-2s + 1) e^{-4s}
$$

The non-invertible part $\bar{G}_m(s)$ contains the RHP zero and the time
delay. As before, the delay is kept separate from the rational part
$\bar{G}_{m,0}(s) = -2s + 1$.

In [ ]:
# Model of the plant (perfect model, rational part)
G_m0 = G_p0

# Invertible part
G_m_inv = K_p / ((T_1 * s + 1) * (T_2 * s + 1))

# Non-invertible part (rational part)
G_m_ninv0 = -T_z * s + 1

# Check that the two parts recombine to the model rational part
diff = con.minreal(G_m_inv * G_m_ninv0 - G_m0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(G_m_inv)
print(G_m_ninv0)

### Feedforward and Reference Model

The feedforward controller and the resulting tracking dynamics are:

$$
G_{ff}(s) = \frac{F(s)}{\tilde{G}_m(s)}
= \frac{(10s + 1)(5s + 1)}{3(5s + 1)^2}
= \frac{10s + 1}{3(5s + 1)}
$$

$$
G_r(s) = F(s) \bar{G}_m(s) = \frac{-2s + 1}{(5s + 1)^2} e^{-4s}
$$

$G_r(s)$ is the reference model: the setpoint response obtained with the
feedforward alone when the model is perfect, $G_r(s) = G_{ff}(s) G_m(s)$.
The feedback controller acts on the error between this reference
trajectory and the plant output.

In [ ]:
# Feedforward (minreal cancels the common (5s + 1) factor)
G_ff = con.minreal(F / G_m_inv, verbose=False)
print(G_ff)

In [ ]:
# Reference model (rational part)
G_r0 = F * G_m_ninv0
print(G_r0)

In [ ]:
# Check that G_ff * G_m0 equals the reference model rational part
diff = con.minreal(G_ff * G_m0 - G_r0, verbose=False)
assert np.allclose(diff.num[0][0], 0.0)
print(con.minreal(G_ff * G_m0, verbose=False))

## Discretization

All blocks are converted to discrete time with a zero-order hold (ZOH)
at sample time $T_s$. The time delay is then an exact shift of
$n_d = \theta / T_s$ samples:

$$
G_p(z) = G_{p,0}(z) \, z^{-n_d}, \qquad G_r(z) = G_{r,0}(z) \, z^{-n_d}
$$

The ZOH conversion is exact for blocks whose input is constant between
samples (e.g. the setpoint path when $r(t)$ steps at a sample time). In
the feedback loop the results are an approximation of the continuous
system, with errors that decrease with $T_s$ (about 0.003 in $y(t)$ for
$T_s = 0.1$ s).

In [ ]:
# Sample time
Ts = 0.1

# Time delay as a shift of n_d samples
n_d = round(theta / Ts)
assert np.isclose(n_d * Ts, theta), "delay must be a multiple of Ts"
z_delay = con.ss(con.tf([1], [1] + [0] * n_d, Ts))

# Discrete-time blocks (ZOH)
G_ff_d = con.c2d(G_ff, Ts, 'zoh')
G_r_d = con.ss(con.c2d(G_r0, Ts, 'zoh')) * z_delay
G_c_d = con.c2d(G_c, Ts, 'zoh')
G_p_d = z_delay * con.ss(con.c2d(G_p0, Ts, 'zoh'))

print(f"Delay: {n_d} samples")
print(G_c_d)

## Closed-Loop System

Each block is given named input and output signals, and the blocks are
connected with `con.interconnect` according to the Simulink diagram:

```none
                 ┌─────────┐  u_ff
         ┌──────►│ G_ff(s) │───────────────────────────────────┐
         │       └─────────┘                                   │                       d
         │                                                     │                       │
         │       ┌─────────┐ r_m        e   ┌──────┐ u_fb      ▼+ u  ┌──────┐          ▼+
r ───────┴──────►│ G_r(s)  │───────►(Σ)────►│G_c(s)│─────────►(Σ)───►│G_p(s)│────────►(Σ)──┬───► y
                 └─────────┘       + ▲−     └──────┘         +       └──────┘        +     │
                                     │                                                     │
                                     └─────────────────────────────────────────────────────┘
```

The disturbance $d$ is added at the plant output.

In [ ]:
# Blocks with named signals
blocks = [
    con.ss(G_ff_d, inputs='r', outputs='u_ff', name='G_ff'),
    con.ss(G_r_d, inputs='r', outputs='r_m', name='G_r'),
    con.ss(G_c_d, inputs='e', outputs='u_fb', name='G_c'),
    con.ss(G_p_d, inputs='u', outputs='y_p', name='G_p'),
    con.summing_junction(inputs=['r_m', '-y'], output='e', dt=Ts, name='sum_e'),
    con.summing_junction(inputs=['u_ff', 'u_fb'], output='u', dt=Ts, name='sum_u'),
    con.summing_junction(inputs=['y_p', 'd'], output='y', dt=Ts, name='sum_y'),
]

# Closed-loop system with inputs (r, d)
T = con.interconnect(
    blocks, inputs=['r', 'd'], outputs=['r_m', 'e', 'u', 'y'],
    name='closed_loop'
)
print(f"Closed-loop system: {T.nstates} states, Ts = {T.dt} s")

## Simulation

Simulation parameters from the Simulink model:

| Parameter | Value |
|---|---|
| Simulation time | 0 – 200 s |
| Setpoint step $r(t)$ | 0 → 1 at $t = 1$ s |
| Disturbance step $d(t)$ | 0 → 1 at $t = 100$ s |
| Initial conditions | zero |

Simulink uses a variable-step solver (RelTol = 1e-3) with exact transport
delays. Here the discrete-time closed-loop system is simulated at the
sample times $t = k T_s$.

In [ ]:
# Simulation parameters
t_stop = 200.0
t_r, r_step = 1.0, 1.0      # setpoint step time and size
t_d, d_step = 100.0, 1.0    # disturbance step time and size

t = Ts * np.arange(round(t_stop / Ts) + 1)
r = np.where(t >= t_r - Ts / 2, r_step, 0.0)
d = np.where(t >= t_d - Ts / 2, d_step, 0.0)

sim = con.forced_response(T, T=t, U=[r, d])
y = sim.outputs[T.find_output('y')]
e = sim.outputs[T.find_output('e')]
u = sim.outputs[T.find_output('u')]
r_m = sim.outputs[T.find_output('r_m')]

In [ ]:
fig, axes = plt.subplots(3, 1, sharex=True, figsize=(7, 1 + 3*1.5))

ax = axes[0]
ax.plot(t, y, label='$y(t)$')
ax.plot(t, r, label='$r(t)$')
ax.plot(t, r_m, '--', label='$r_m(t)$')
ax.set_ylabel('$r(t)$, $y(t)$')
ax.set_title('Anticipation de la consigne')
ax.legend()

ax = axes[1]
ax.plot(t, e)
ax.set_ylabel('$e(t)$')

ax = axes[2]
ax.plot(t, u)
ax.set_ylabel('$u(t)$')
ax.set_xlabel('Temps (s)')

for ax in axes:
    ax.grid()
plt.tight_layout()
plt.show()